In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Data Analytics: A2A HTTP API Sample

This notebook demonstrates how to interact with the **DataA2AService** using standard HTTP requests. This is useful for environments where a high-level SDK is not available or when you want to minimize dependencies.

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/agents/gemini_data_analytics/a2a_http_sample.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fagents%2Fgemini_data_analytics%2Fa2a_http_sample.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/agents/gemini_data_analytics/a2a_http_sample.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/gemini_data_analytics/a2a_http_sample.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>


# Background and Overview
The **Conversational Analytics API** (also known as Gemini Data Analytics) lets you chat with your BigQuery or Looker data anywhere. This notebook demonstrates how to use the **A2A (Agent-to-Agent)** interface via standard HTTP requests. This is useful for environments where a high-level SDK is not available or when you want to minimize dependencies.

This is a **Pre-GA** product. See [documentation](https://cloud.google.com/gemini/docs/conversational-analytics-api/overview) for more details.

Please provide feedback to conversational-analytics-api-feedback@google.com
<br>
### This notebook will help you
1. Authenticate to Google Cloud
2. Retrieve the Agent Card
3. Send asynchronous messages and poll for results
4. Process agent outputs (Artifacts)
5. Cancel active tasks


In [ ]:
# @title Setup and Authentication
import json
import os
import time
import uuid
from google.auth import default
from google.auth.transport.requests import Request
from google.colab import auth
import requests

# Authenticate the user
auth.authenticate_user()

# Get credentials and project ID
creds, _ = default()
creds.refresh(Request())
access_token = creds.token

ENDPOINT = "https://geminidataanalytics.googleapis.com"
LOCATION = "global" # @param {type:"string"}
PROJECT_ID = "[your-project-id]" # @param {type:"string"}
# AGENT_ID can be found from the Cloud URL, e.g.
# https://console.cloud.google.com/bigquery/agents_hub/<your-agent-id>?project=<your-project-id>
AGENT_ID = "your-agent-id" # @param {type:"string"}

if not PROJECT_ID or PROJECT_ID == "[your-project-id]"
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))
if not LOCATION:
    LOCATION = os.environ.get("GOOGLE_CLOUD_REGION")
    
TENANT = f"projects/{PROJECT_ID}/locations/{LOCATION}/agents/{AGENT_ID}"
BASE_URL = f"{ENDPOINT}/v1beta/a2a/{TENANT}/v1"
HEADERS = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
}
print(f"Target Tenant: {TENANT}")
print(f"Access Token Length: {len(access_token) if access_token else 0}")


## 1. Get Agent Card

First, let's retrieve the Agent Card to verify connectivity and see what the agent can do.

In [ ]:
url = f"{BASE_URL}/card"

try:
  response = requests.get(url, headers=HEADERS, timeout=30)
  response.raise_for_status()
  print("Agent Card:")
  print(json.dumps(response.json(), indent=2))
except Exception as e:
  print(f"Error fetching agent card: {e}")

## 2. Send Message (Asynchronous + Polling)

For long-running tasks, use `blocking=False`. This returns a `Task` object immediately, which you can poll for status.

In [ ]:
USER_QUERY = "Hello" # @param {type:"string"}

def send_async_message(query):
  url = f"{BASE_URL}/message:send"
  payload = {
      "tenant": TENANT,
      "message": {
          "message_id": f"msg-{uuid.uuid4()}",
          "role": "ROLE_USER",
          "content": [{"text": query}],
      },
      "configuration": {"blocking": False},
  }

  response = requests.post(url, headers=HEADERS, json=payload, timeout=30)
  response.raise_for_status()

  res_json = response.json()
  
  task = res_json.get("task")
  if task:
    print(f"Task created: {task.get('id')}")
    return task.get("id")
  elif "message" in res_json:
    print("Received message directly instead of task.")
    return None
  else:
    print("Response did not contain 'task' or 'message'")
    return None


def poll_task(task_id, max_retries=15):
  url = f"{BASE_URL}/tasks/{task_id}"
  retry_count = 0
  wait_time = 2

  while retry_count < max_retries:
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()

    task = response.json()
    state = task.get("status", {}).get("state")
    print(f"Current State: {state}")

    if state in [
        "TASK_STATE_COMPLETED",
        "TASK_STATE_FAILED",
        "TASK_STATE_CANCELLED",
    ]:
      return task

    time.sleep(wait_time)
    wait_time = min(wait_time * 1.5, 10)
    retry_count += 1

  print("Polling timed out.")
  return None

try:
  print("Starting Send Message...")
  task_id = send_async_message(USER_QUERY)
  
  if task_id:
    print(f"Polling task {task_id}...")
    final_task = poll_task(task_id)
    if final_task:
      print("Final Task Result:")
      print(json.dumps(final_task, indent=2))
  else:
    print("No task ID returned, skipping polling.")
    
except Exception as e:
  print(f"Error during messaging/polling: {e}")

## 3. Processing Agent Outputs (Artifacts)

Agents often produce **Artifacts** (structured data, files, or references). Here is how to parse them from a completed task.

In [ ]:
USER_QUERY = "Which item categories had the highest sales last year?" # @param {type:"string"}


def get_data_and_extract_artifacts(query):
  url = f"{BASE_URL}/message:send"
  payload = {
      "tenant": TENANT,
      "message": {
          "message_id": f"msg-{uuid.uuid4()}",
          "role": "ROLE_USER",
          "content": [{"text": query}],
      },
      # We set blocking=True to wait for the full response (including artifacts)
      "configuration": {"blocking": True},
  }

  print(f"Sending analysis query: '{query}'")
  print("This involves data processing, so it may take 30-60 seconds...")
  
  # 120 second timeout to give it plenty of time to compute
  response = requests.post(url, headers=HEADERS, json=payload, timeout=120)
  response.raise_for_status()

  res_json = response.json()
  print("\n--- Response Received ---")

  # Case 1: The server returned a Task
  task = res_json.get("task")
  if task:
    print("Server returned a Task. Processing artifacts...")
    artifacts = task.get("artifacts", [])
    display_artifacts(artifacts)
    return

  # Case 2: The server returned a Direct Message
  message = res_json.get("message")
  if message:
    print("Server returned a direct Message.")
    # Check if there are artifacts attached to the message or in the content
    content_parts = message.get("content", [])
    
    print(f"Message contains {len(content_parts)} content parts.")
    for part in content_parts:
      # Print the text response
      if "text" in part:
        print(f"\nText Response:\n{part['text']}")
      
      # Check for metadata that might contain structured data (artifacts)
      metadata = part.get("metadata", {})
      if metadata:
        print(f"\nMetadata found: {json.dumps(metadata, indent=2)}")
    return

  print(f"Unexpected response structure: {res_json}")


def display_artifacts(artifacts):
  if not artifacts:
    print("No artifacts found in the task.")
    return

  print(f"Found {len(artifacts)} artifacts:")
  for art in artifacts:
    name = art.get("name", "Unnamed")
    
    art_type = (
        art.get("metadata", {})
        .get("fields", {})
        .get("type", {})
        .get("stringValue", None)
    )
    
    if not art_type:
        art_type = art.get("metadata", {}).get("type", "Unknown")
        
    print(f"\n- [{art_type}] {name}: {art.get('description')}")
    
    for part in art.get("parts", []):
      if "text" in part:
        content = part["text"]
        snippet = content[:500] + "..." if len(content) > 500 else content
        print(f"  Content:\n{snippet}")

try:
  get_data_and_extract_artifacts(USER_QUERY)
except Exception as e:
  print(f"Error during artifact extraction: {e}")

## 4. Cancel an Active Task

If a task is taking too long or was sent in error, you can cancel it.

In [ ]:
USER_QUERY = "Which item categories had the highest sales last year?" # @param {type:"string"}

def cancel_task(task_id):
  url = f"{BASE_URL}/tasks/{task_id}:cancel"
  response = requests.post(url, headers=HEADERS, timeout=30)
  response.raise_for_status()

  print(f"Task {task_id} cancellation requested.")
  return response.json()

def send_async_message_with_timeout(query, timeout_secs=60):
  url = f"{BASE_URL}/message:send"
  payload = {
      "tenant": TENANT,
      "message": {
          "message_id": f"msg-{uuid.uuid4()}",
          "role": "ROLE_USER",
          "content": [{"text": query}],
      },
      "configuration": {"blocking": False},
  }

  response = requests.post(url, headers=HEADERS, json=payload, timeout=timeout_secs)
  response.raise_for_status()
  
  res_json = response.json()
  task = res_json.get("task")
  if task:
      return task.get("id")
  return None

try:
  print(f"Starting task with 60s timeout: '{USER_QUERY}'")
  
  new_task_id = send_async_message_with_timeout(USER_QUERY, timeout_secs=60)
  
  if new_task_id:
    print(f"Task created! ID: {new_task_id}. Cancelling now...")
    cancel_response = cancel_task(new_task_id)
    print(f"Cancel Response: {cancel_response}")
  else:
    print("No task ID returned (the server might have answered immediately).")
    
except Exception as e:
  print(f"Error during cancellation demo: {e}")

## 5. Cleanup

It is good practice to clean up any temporary resources or state created during your session.

In [ ]:
# @title Resource Cleanup
print(
    "No specific cloud resources were created in this demo that require manual"
    " deletion (e.g., storage buckets)."
)
print(
    "However, you can use this section to reset any local session state if"
    " needed."
)